In [1]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = pd.read_csv('supplementary_data/data.csv')
feature_columns = ["O","N","SSA","PV","RMIC","Dap","ID/IG","CD","Anion"]
target = "Cs" # 比容量


data["Anion"] = data["Anion"].replace({
    "SO4": 0,
    "OTf": 1
}).astype(int)

X = data[feature_columns]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))


X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).reshape(-1, 1)
X_test_tensor  = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test_scaled,  dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)


C:\Users\luluz\AppData\Local\Temp\ipykernel_29672\1203554300.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["Anion"] = data["Anion"].replace({


In [2]:
import random
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

seed = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = pd.read_csv("supplementary_data/data.csv")

feature_columns = [
    "O", "N", "SSA", "PV", "RMIC",
    "Dap", "ID/IG", "CD", "Anion"
]
target = "Cs"

data["Anion"] = data["Anion"].map({
    "SO4": 0,
    "OTf": 1
}).astype(int)

X = data[feature_columns]
y = data[target]


X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed,
)


X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15,
    random_state=seed,
)

x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1))

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).reshape(-1, 1)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32).reshape(-1, 1).to(device)

train_dataset = TensorDataset(X_train_tensor,y_train_tensor)

# Hyper Parameters
batch_size = 20
input_size = 9
hidden_layers = [120, 40, 50, 120]
output_size = 1
learning_rate = 0.002
num_epochs = 200
patience = 30

generator = torch.Generator()
generator.manual_seed(seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

class ANNRegressor(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size):
        super(ANNRegressor, self).__init__()
        layers = []
        prev_size = input_size

        for layer in hidden_layers:
            layers.append(nn.Linear(prev_size, layer))
            layers.append(nn.ReLU())
            prev_size = layer

        layers.append(nn.Linear(prev_size, output_size))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = ANNRegressor(input_size, hidden_layers, output_size).to(device)

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

best_val_loss = np.inf
best_val_r2 = -np.inf
best_epoch = 0
best_state = None
counter = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()

    with torch.no_grad():
        val_preds = model(X_val_tensor)
        val_loss = criterion(val_preds, y_val_tensor).item()

    val_preds_real = y_scaler.inverse_transform(val_preds.cpu().numpy())
    val_labels_real = y_scaler.inverse_transform(y_val_tensor.cpu().numpy())

    val_r2 = r2_score(val_labels_real, val_preds_real)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_r2 = val_r2
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1}/{num_epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val R2: {val_r2:.4f} | "
            f"Best R2: {best_val_r2:.4f}"
        )

    if counter >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

print(f"Best Epoch: {best_epoch}")
print(f"Best Validation R2: {best_val_r2:.4f}")

x_scaler = StandardScaler()
X_train_full_scaled = x_scaler.fit_transform(X_train_full)
X_test_scaled = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_full_scaled = y_scaler.fit_transform(y_train_full.values.reshape(-1, 1))

X_train_full_tensor = torch.tensor(X_train_full_scaled,dtype=torch.float32)
y_train_full_tensor = torch.tensor(y_train_full_scaled,dtype=torch.float32).reshape(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32).to(device)

train_full_dataset = TensorDataset(
    X_train_full_tensor,
    y_train_full_tensor
)

set_seed(seed)

generator = torch.Generator()
generator.manual_seed(seed)

train_full_loader = DataLoader(
    train_full_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

model = ANNRegressor(
    input_size,
    hidden_layers,
    output_size
).to(device)

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

for epoch in range(best_epoch):
    model.train()

    for batch_X, batch_y in train_full_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

torch.save(
    model.state_dict(),
    "best_ann_model.pth"
)

model.eval()

with torch.no_grad():
    train_preds_scaled = model(
        X_train_full_tensor.to(device)
    ).cpu().numpy()

    test_preds_scaled = model(
        X_test_tensor
    ).cpu().numpy()

train_preds = y_scaler.inverse_transform(
    train_preds_scaled
).ravel()

test_preds = y_scaler.inverse_transform(
    test_preds_scaled
).ravel()

y_train_true = y_train_full.values
y_test_true = y_test.values

train_r2 = r2_score(
    y_train_true,
    train_preds
)
test_r2 = r2_score(
    y_test_true,
    test_preds
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train_true,
        train_preds
    )
)
test_rmse = np.sqrt(
    mean_squared_error(
        y_test_true,
        test_preds
    )
)

train_mae = mean_absolute_error(
    y_train_true,
    train_preds
)
test_mae = mean_absolute_error(
    y_test_true,
    test_preds
)

train_mape = np.mean(
    np.abs(
        (y_train_true - train_preds) /
        y_train_true
    )
) * 100

test_mape = np.mean(
    np.abs(
        (y_test_true - test_preds) /
        y_test_true
    )
) * 100

print("-------- ANN --------")
print(f"Train set R2   : {train_r2:.4f}")
print(f"Test set R2    : {test_r2:.4f}")
print(f"Train set RMSE : {train_rmse:.4f}")
print(f"Test set RMSE  : {test_rmse:.4f}")
print(f"Train set MAE  : {train_mae:.4f}")
print(f"Test set MAE   : {test_mae:.4f}")
print(f"Train set MAPE : {train_mape:.2f}%")
print(f"Test set MAPE  : {test_mape:.2f}%")

Epoch [10/200] | Train Loss: 0.2484 | Val Loss: 0.2705 | Val R2: 0.7553 | Best R2: 0.7595
Epoch [20/200] | Train Loss: 0.1593 | Val Loss: 0.2304 | Val R2: 0.7916 | Best R2: 0.8226
Epoch [30/200] | Train Loss: 0.1372 | Val Loss: 0.2050 | Val R2: 0.8145 | Best R2: 0.8286
Epoch [40/200] | Train Loss: 0.1228 | Val Loss: 0.2078 | Val R2: 0.8120 | Best R2: 0.8731
Epoch [50/200] | Train Loss: 0.0892 | Val Loss: 0.1345 | Val R2: 0.8784 | Best R2: 0.8784
Epoch [60/200] | Train Loss: 0.1092 | Val Loss: 0.1349 | Val R2: 0.8780 | Best R2: 0.8838
Epoch [70/200] | Train Loss: 0.0715 | Val Loss: 0.1280 | Val R2: 0.8842 | Best R2: 0.8914
Epoch [80/200] | Train Loss: 0.0841 | Val Loss: 0.1177 | Val R2: 0.8936 | Best R2: 0.8936
Epoch [90/200] | Train Loss: 0.0739 | Val Loss: 0.1324 | Val R2: 0.8802 | Best R2: 0.9043
Epoch [100/200] | Train Loss: 0.1067 | Val Loss: 0.1286 | Val R2: 0.8837 | Best R2: 0.9043
Epoch [110/200] | Train Loss: 0.0804 | Val Loss: 0.1060 | Val R2: 0.9041 | Best R2: 0.9043
Epoch [1